## Actividad 3_10

<div style="border-style:groove;border-width:thin;padding:10px">

En esta actividad vamos a tratar de solucionar un problema mediante clasificación, aunque también podría tratarse como regresión. 
Se trata de un dataset de valoración de vinos. 

<p style="border-style:groove;border-width:thin;padding:10px">
Lo primero que vamos a hacer es importar los datos y analizar el dataset que tenemos.
</p>

In [30]:
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.tree import export_graphviz
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
wines = pd.read_csv('winequalityN.csv')
wines.head()

,type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,white,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6.0
1,white,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6.0
2,white,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6.0
3,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6.0
4,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6.0


In [31]:
wines['quality'].unique()

array([ 6.,  5., nan,  7.,  8.,  4.,  3.,  9.])

Características del dataset:
<ul>
<li>La columna quality nos da la puntuación de calidad que le dan al vino. Queremos predecirla en función de las características del vino. </li>
<li>Tenemos, además de nulos, 7 posibles valores. Se podría hacer como un problema de regresión o como uno de clasificación de 7 clases.</li>
</ul>
</div>

Prepara el dataset para poder resolverlo con los dos algoritmos de clasificación que hemos visto, decision tree y SVC. Cuidado con los nulos, trátalos de la forma adecuada. Haz la correlación de las variables con la variable objetivo para ir intuyendo lo que podremos conseguir. Trabaja con los hiperparámetros para conseguir el mejor resultado posible. Valora los resultados y haz los cambios que consideres oportunos.

In [32]:
wines.dropna(inplace=True)
wines = pd.get_dummies(wines, columns=['type'],dtype=int)
wines.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6445 entries, 0 to 6496
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         6445 non-null   float64
 1   volatile acidity      6445 non-null   float64
 2   citric acid           6445 non-null   float64
 3   residual sugar        6445 non-null   float64
 4   chlorides             6445 non-null   float64
 5   free sulfur dioxide   6445 non-null   float64
 6   total sulfur dioxide  6445 non-null   float64
 7   density               6445 non-null   float64
 8   pH                    6445 non-null   float64
 9   sulphates             6445 non-null   float64
 10  alcohol               6445 non-null   float64
 11  quality               6445 non-null   float64
 12  type_red              6445 non-null   int64  
 13  type_white            6445 non-null   int64  
dtypes: float64(12), int64(2)
memory usage: 755.3 KB


In [33]:
wines.corr(numeric_only=True)['quality'].abs().sort_values(ascending=False)[1:]

alcohol                 0.444986
density                 0.304342
volatile acidity        0.265735
chlorides               0.199723
type_white              0.118791
type_red                0.118791
citric acid             0.084799
fixed acidity           0.075812
free sulfur dioxide     0.054267
total sulfur dioxide    0.041668
sulphates               0.039172
residual sugar          0.034995
pH                      0.017710
Name: quality, dtype: float64

In [ ]:
X = wines.drop(['quality'],axis=1)
y = wines['quality'].to_frame()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [65]:
tree_clf = DecisionTreeClassifier()
tree_clf.fit(X_train, y_train)

export_graphviz(
    tree_clf,
    out_file="./wines_tree.dot",
    feature_names=list(X_train.columns),
    class_names=[str(c) for c in tree_clf.classes_],
    rounded=True,
    filled=True
)

# If "dot: command not found" occurs, install graphviz in the system:
# sudo apt install graphviz
!dot -Tpng wines_tree.dot -o wines_tree.png

dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.251304 to fit


In [66]:
y_pred = tree_clf.predict(X_test)
print(y_pred)
print("Accuracy:", accuracy_score(y_test, y_pred))

[6. 7. 6. ... 6. 7. 5.]
Accuracy: 0.6183087664856478


In [37]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [52]:
#Entrenamiento con diferentes kernels para sacar la mejor configuración
valores_C = [0.1, 1, 4, 7, 10]
valores_degree = [2, 3, 4, 5, 6]

mejor_acc = 0
mejor_config = None

for C in valores_C:
    for degree in valores_degree:
        print(f"\nProbando C={C} | degree={degree}")

        svm_poly = SVC(kernel="rbf", degree=degree, coef0=1, C=C)
        svm_poly.fit(X_train_scaled, y_train)

        pred = svm_poly.predict(X_test_scaled)
        acc = accuracy_score(y_test, pred)

        print(f"Accuracy: {acc:.4f}")

        if acc > mejor_acc:
            mejor_acc = acc
            mejor_config = (C, degree)

pred_rbf = svm_poly.predict(X_test_scaled)

print(f"Mejor Accuracy: {mejor_acc:.4f}")
print(f"Mejor C y degree: C={mejor_config[0]}, degree={mejor_config[1]}")
print(confusion_matrix(y_test, pred_rbf))
print(classification_report(y_test, pred_rbf))


Probando C=0.1 | degree=2


/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Accuracy: 0.5423

Probando C=0.1 | degree=3


/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Accuracy: 0.5423

Probando C=0.1 | degree=4


...

/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Accuracy: 0.5997

Probando C=10 | degree=6


/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Accuracy: 0.5997
Mejor Accuracy: 0.6082
Mejor C y degree: C=4, degree=2
[[  0   1   3   2   0   0   0]
 [  0   5  28   8   1   0   0]
 [  0   4 279 139   3   0   0]
 [  0   2 120 413  28   0   0]
 [  0   0   4 136  74   0   0]
 [  0   0   0  28   8   2   0]
 [  0   0   0   1   0   0   0]]
              precision    recall  f1-score   support

         3.0       0.00      0.00      0.00         6
         4.0       0.42      0.12      0.19        42
         5.0       0.64      0.66      0.65       425
         6.0       0.57      0.73      0.64       563
         7.0       0.65      0.35      0.45       214
         8.0       1.00      0.05      0.10        38
         9.0       0.00      0.00      0.00         1

    accuracy                           0.60      1289
   macro avg       0.47      0.27      0.29      1289
weighted avg       0.61      0.60      0.58      1289



/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [54]:
#Mejor configuración encontrada después de haber probado el rbs y el lineal, es el polinomial
svm_poly = SVC(kernel="rbf", degree=2, coef0=1, C=5)
svm_poly.fit(X_train_scaled, y_train)
pred_poly = svm_poly.predict(X_test_scaled)

print("=== SVM RBF ===")
print("Accuracy:", accuracy_score(y_test, pred_poly))
print(confusion_matrix(y_test, pred_poly))
print(classification_report(y_test, pred_poly))

/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


=== SVM RBF ===
Accuracy: 0.604344453064391
[[  0   1   3   2   0   0   0]
 [  0   4  29   7   2   0   0]
 [  0   3 275 145   2   0   0]
 [  0   1 115 427  20   0   0]
 [  0   0   4 137  73   0   0]
 [  0   0   0  32   6   0   0]
 [  0   0   0   1   0   0   0]]
              precision    recall  f1-score   support

         3.0       0.00      0.00      0.00         6
         4.0       0.44      0.10      0.16        42
         5.0       0.65      0.65      0.65       425
         6.0       0.57      0.76      0.65       563
         7.0       0.71      0.34      0.46       214
         8.0       0.00      0.00      0.00        38
         9.0       0.00      0.00      0.00         1

    accuracy                           0.60      1289
   macro avg       0.34      0.26      0.27      1289
weighted avg       0.59      0.60      0.58      1289



/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
